In [ ]:
pip install requests transformers datasets einops peft scikit-learn xgboost rdkit

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import requests

In [ ]:
import torch
import json, requests

from transformers import AutoModel, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

hf_cache = "/scratch/sguirale/tmp"

# Structure-derived vs structure-free

One dataset, one set of physical features, two models. The **only** difference
between them is the HADDOCK3 block:

    structure-free    embeddings + RDKit descriptors + ECFP4 + AA composition
    structure-based   the same, plus HADDOCK3 metrics

`iptm` is excluded from both. It comes from an AlphaFold3 run, and a user
supplying a complex may not have it, so a model that depends on it is not
deployable.

## Get Training Data

In [ ]:
## HADDOCK3 metrics. iptm is deliberately absent - see the note above.
HADDOCK_COLS = ["score", "total", "vdw", "elec", "desolv", "bsa",
                "num_interacting_residues"]

raw_df = pd.read_csv("combined_50k_filtered.csv")[
    ["seq", "smiles", "neg_log10_affinity_M"] + HADDOCK_COLS]

raw_df.head()

In [ ]:
## Other Filters

raw_df = raw_df[raw_df['neg_log10_affinity_M'] >= 4]

print(f"Filtered dataset size: {len(raw_df)}")

In [ ]:
## Drop ligands RDKit cannot parse, so every feature block has the same rows.
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

raw_df = raw_df[[Chem.MolFromSmiles(s) is not None for s in raw_df['smiles']]].reset_index(drop=True)
print(f"Parseable rows: {len(raw_df)}")

In [ ]:
## About 0.9% of the docking runs failed: vdw reaches 1.5e7 where a physically
## reasonable term is roughly -200 to +100, i.e. clashing poses rather than
## energies. Their pKd is normal (6.50 vs 6.67), so this is bad docking, not bad
## binders. The failures form a separate cluster - the next largest vdw below the
## cutoff is around 88 - so a flat threshold separates them cleanly. Dropping
## beats clipping here: 0.9% is above the 99.9th percentile, so quantile clipping
## would squash them to a bound still in the millions. Both models see the same
## rows either way, so the comparison stays controlled.
failed = raw_df["vdw"] > 1000

print(f"Dropping {int(failed.sum())} failed docking runs ({failed.mean():.1%}); "
      f"largest surviving vdw is {raw_df.loc[~failed, 'vdw'].max():.1f}")
raw_df = raw_df[~failed].reset_index(drop=True)
print(raw_df[HADDOCK_COLS].describe().T[["mean", "std", "min", "max"]])

## Load Protein Tokenizer

In [ ]:
## Load Protein Models
base_model_path = "Synthyra/ESM2-8M"

esm2_model = AutoModel.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    attn_implementation="sdpa",
    cache_dir=hf_cache
).to(device).eval()


esm2_tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    cache_dir=hf_cache
)

In [ ]:
## Function to get sequence embeddings
@torch.inference_mode()
def get_sequence_embeddings(model, sequences, batch_size=4):
    embedding_batches = []

    for start in range(0, len(sequences), batch_size):
        sequence_batch = sequences[start:start + batch_size]
        batch = esm2_tokenizer(
            sequence_batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)

        outputs = model(**batch)
        hidden = outputs.last_hidden_state

        ## Ignore padding during mean pooling.
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        embedding_batches.append(pooled.cpu())

        del batch, outputs, hidden, mask, pooled
        if device == "cuda":
            torch.cuda.empty_cache()

    return torch.cat(embedding_batches, dim=0)

In [ ]:
aa_sequences = raw_df['seq'].tolist()

In [ ]:
aa_emb = get_sequence_embeddings(esm2_model, aa_sequences)

## Get Ligand Tokenizer

In [ ]:
ligand_model_name = "DeepChem/ChemBERTa-77M-MLM"

ligand_tokenizer = AutoTokenizer.from_pretrained(ligand_model_name, cache_dir=hf_cache)
ligand_model = AutoModel.from_pretrained(ligand_model_name, cache_dir=hf_cache).to(device).eval()

In [ ]:
## Function to get SMILES embeddings
@torch.inference_mode()
def get_smiles_embeddings(model, smiles, batch_size=32):
    embedding_batches = []

    for start in range(0, len(smiles), batch_size):
        smiles_batch = smiles[start:start + batch_size]
        batch = ligand_tokenizer(
            smiles_batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)

        outputs = model(**batch)
        hidden = outputs.last_hidden_state

        ## Ignore padding during mean pooling.
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        embedding_batches.append(pooled.cpu())

        del batch, outputs, hidden, mask, pooled
        if device == "cuda":
            torch.cuda.empty_cache()

    return torch.cat(embedding_batches, dim=0)

In [ ]:
smiles_sequences = raw_df['smiles'].tolist()

In [ ]:
lig_emb = get_smiles_embeddings(ligand_model, smiles_sequences)

## Physical Features

These go into **both** models. Everything here comes from the sequence and the
SMILES alone - no folding, no docking, CPU-only.

In [ ]:
## RDKit ligand descriptors. AddHs reproduces the 11 columns in the training csv
## exactly (checked on all 32,086 unique ligands); on the bare molecule
## num_h_acceptors and num_rotatable_bonds come out wrong.
from rdkit.Chem import Crippen, Descriptors, Lipinski, rdFingerprintGenerator, rdMolDescriptors

mols = [Chem.MolFromSmiles(s) for s in raw_df['smiles']]
mols_h = [Chem.AddHs(m) for m in mols]

desc11 = np.array([[Descriptors.MolWt(m), Crippen.MolLogP(m), Lipinski.NumHDonors(m),
                    Lipinski.NumHAcceptors(m), Lipinski.NumRotatableBonds(m),
                    rdMolDescriptors.CalcTPSA(m), rdMolDescriptors.CalcNumRings(m),
                    rdMolDescriptors.CalcNumAromaticRings(m), m.GetNumAtoms(),
                    m.GetNumHeavyAtoms(), Chem.GetFormalCharge(m)]
                   for m in mols_h], dtype=np.float32)
print("RDKit 11:", desc11.shape)

In [ ]:
## The full RDKit descriptor set - a few hundred more numbers, all free.
## Cast to float32 first: Kappa2 and friends return values so large they overflow
## to inf only on the cast, so cleaning before it leaves infs XGBoost rejects.
desc_full = pd.DataFrame([Descriptors.CalcMolDescriptors(m) for m in mols]).astype(np.float32)
desc_full = desc_full.replace([np.inf, -np.inf], np.nan).fillna(0.0)
desc_full = desc_full.loc[:, desc_full.std() > 0]

assert np.isfinite(desc_full.to_numpy()).all(), "non-finite values remain in desc_full"
desc_full = desc_full.to_numpy(np.float32)
print("RDKit full:", desc_full.shape)

In [ ]:
## ECFP4 count fingerprint. Kept as a separate block so it can be switched off:
## exact-substructure bits are what let the model memorise chemotypes, and the
## benchmark gates ligands at Tanimoto 0.50, where those bits stop transferring.
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
ecfp = np.array([morgan.GetCountFingerprintAsNumPy(m) for m in mols], dtype=np.float32)
print("ECFP4:", ecfp.shape)

In [ ]:
## Amino-acid composition and length: a cheap protein signal that mean-pooled
## ESM2 largely washes out.
AA = "ACDEFGHIKLMNPQRSTVWY"
aa_comp = np.array([[s.count(a) / len(s) for a in AA] + [len(s)]
                    for s in raw_df['seq']], dtype=np.float32)
print("AA composition:", aa_comp.shape)

In [ ]:
## Murcko scaffolds, for a split that holds out whole chemotypes rather than
## exact molecules. Acyclic ligands have no scaffold, so they fall back to their
## own canonical SMILES and end up grouped by identity.
from rdkit.Chem.Scaffolds import MurckoScaffold

def scaffold_of(m):
    s = MurckoScaffold.MurckoScaffoldSmiles(mol=m)
    return s if s else Chem.MolToSmiles(m)

scaffolds = np.array([scaffold_of(m) for m in mols])
print(f"unique ligands:   {raw_df['smiles'].nunique()}")
print(f"unique scaffolds: {len(np.unique(scaffolds))}")

In [ ]:
## Four feature sets: the two tiers, each with and without ECFP.
X_emb = np.concatenate([lig_emb.numpy(), aa_emb.numpy()], axis=1)
haddock = raw_df[HADDOCK_COLS].to_numpy(np.float32)
y = raw_df['neg_log10_affinity_M'].to_numpy(np.float32)

X_free = np.concatenate([X_emb, desc11, desc_full, aa_comp], axis=1)
X_free_ecfp = np.concatenate([X_free, ecfp], axis=1)

FEATURES = {
    "free": X_free,
    "free+ecfp": X_free_ecfp,
    "struct": np.concatenate([X_free, haddock], axis=1),
    "struct+ecfp": np.concatenate([X_free_ecfp, haddock], axis=1),
}
for name, block in FEATURES.items():
    print(f"{name:14s} {block.shape}")

## Splits

Two changes from the previous run, both aimed at the benchmark failure:

1. **The inner split now uses the same grouping as the outer split.** Previously
   early stopping always grouped by protein, so for a ligand-held-out model the
   validation fold still shared ligands with the fit fold - early stopping was
   rewarding ligand memorisation and letting training run past the point where it
   generalised.
2. **Scaffold-grouped replaces ligand-grouped.** Holding out exact molecules
   still lets close analogues sit on both sides. Holding out Murcko scaffolds
   removes whole chemotypes, which is much closer to the benchmark's Tanimoto
   0.50 gate.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def grouped(groups, test_size=0.25, seed=1337):
    return next(GroupShuffleSplit(1, test_size=test_size, random_state=seed)
                .split(np.zeros((len(groups), 1)), groups=groups))

seqs = raw_df['seq'].to_numpy()

tr_p, te_p = grouped(seqs)
tr_s, te_s = grouped(scaffolds)
## Unseen both: start from the scaffold split, drop test rows on a trained protein.
tr_b = tr_s
te_b = te_s[~np.isin(seqs[te_s], np.unique(seqs[tr_s]))]

## Each split carries the group key that produced it, so the inner
## early-stopping split can hold out the same thing the outer split does.
SPLITS = {"protein-grouped":  (tr_p, te_p, seqs),
          "scaffold-grouped": (tr_s, te_s, scaffolds),
          "both-disjoint":    (tr_b, te_b, scaffolds)}
for name, (tr, te, _) in SPLITS.items():
    print(f"{name:17s} train {len(tr):6d}  test {len(te):6d}")

## Train and Evaluate

In [ ]:
## Same models and hyperparameters as before. The one change is `groups=g[tr]`
## in the inner split - it used to be `groups=seqs[tr]` regardless of the split.
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score,
                             roc_auc_score)
from scipy.stats import spearmanr

BINDER_PKD_THRESHOLD = 6.0
is_binder = (y > BINDER_PKD_THRESHOLD).astype(int)
results, trained = [], {}

for fname, Xf in FEATURES.items():
    for sname, (tr, te, g) in SPLITS.items():
        ## hold out the SAME key the outer split holds out
        i_fit, i_val = next(GroupShuffleSplit(1, test_size=0.2, random_state=1337)
                            .split(tr, y[tr], groups=g[tr]))
        fit, val = tr[i_fit], tr[i_val]

        reg = xgb.XGBRegressor(
            n_estimators=3000, max_depth=8, learning_rate=0.03, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.8, objective='reg:squarederror',
            eval_metric='rmse', early_stopping_rounds=50, random_state=1337
        )
        reg.fit(Xf[fit], y[fit], eval_set=[(Xf[val], y[val])], verbose=False)
        pred = reg.predict(Xf[te])

        clf = HistGradientBoostingClassifier(
            max_iter=400, learning_rate=0.06, l2_regularization=1.0,
            class_weight="balanced", random_state=1337
        ).fit(Xf[tr], is_binder[tr])
        proba = clf.predict_proba(Xf[te])[:, 1]

        results.append({
            "features": fname, "split": sname, "n_test": len(te),
            "best_iteration": int(reg.best_iteration),
            "RMSE": np.sqrt(mean_squared_error(y[te], pred)),
            "MAE": mean_absolute_error(y[te], pred),
            "R2": r2_score(y[te], pred),
            "Pearson": float(np.corrcoef(y[te], pred)[0, 1]),
            "Spearman": spearmanr(y[te], pred).correlation,
            "pred_std_ratio": float(pred.std() / y[te].std()),
            "Accuracy": accuracy_score(is_binder[te], proba >= 0.5),
            "F1": f1_score(is_binder[te], proba >= 0.5, zero_division=0),
            "ROC_AUC": roc_auc_score(is_binder[te], proba),
            "PR_AUC": average_precision_score(is_binder[te], proba),
        })
        trained[(fname, sname)] = (reg, clf)
        r = results[-1]
        print(f"{fname:14s} {sname:17s} Pearson {r['Pearson']:.3f}  R2 {r['R2']:6.3f}  "
              f"spread {r['pred_std_ratio']:.2f}  ROC {r['ROC_AUC']:.3f}  "
              f"iters {r['best_iteration']}")

results_df = pd.DataFrame(results)
results_df

In [ ]:
## Does dropping ECFP help? Positive gain = the no-ECFP model is better.
piv = results_df.pivot(index="split", columns="features", values="Pearson").reindex(list(SPLITS))
piv["free: drop ecfp"] = piv["free"] - piv["free+ecfp"]
piv["struct: drop ecfp"] = piv["struct"] - piv["struct+ecfp"]
piv.round(4)

In [ ]:
## pred_std_ratio is the variance-collapse diagnostic: predictions spanning far
## less than the truth means the model is falling back to the training mean.
## On the benchmark it was 0.34 while Boltz-2 managed 0.61.
results_df.pivot(index="split", columns="features", values="pred_std_ratio").reindex(list(SPLITS)).round(3)

## Compare

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

Path("models_structure_free").mkdir(exist_ok=True)
names = list(SPLITS)
x = np.arange(len(names))
width = 0.8 / len(FEATURES)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, metric in zip(axes.ravel(), ["Pearson", "R2", "RMSE", "pred_std_ratio"]):
    for i, fname in enumerate(FEATURES):
        vals = [results_df.query("features == @fname and split == @s")[metric].iloc[0] for s in names]
        bars = ax.bar(x + i * width, vals, width, label=fname)
        ax.bar_label(bars, fmt="%.2f", fontsize=6)
    ax.set_xticks(x + width * (len(FEATURES) - 1) / 2)
    ax.set_xticklabels(names, fontsize=8)
    ax.set_title(f"{metric} ({'lower better' if metric == 'RMSE' else 'higher better'})", fontsize=10)
axes.ravel()[0].legend(fontsize=7)

fig.suptitle("Regressors: fixed inner split, ECFP on/off", fontsize=12)
fig.tight_layout()
fig.savefig("models_structure_free/regressor_comparison.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, metric in zip(axes.ravel(), ["ROC_AUC", "PR_AUC", "F1", "Accuracy"]):
    for i, fname in enumerate(FEATURES):
        vals = [results_df.query("features == @fname and split == @s")[metric].iloc[0] for s in names]
        bars = ax.bar(x + i * width, vals, width, label=fname)
        ax.bar_label(bars, fmt="%.2f", fontsize=6)
    ax.set_xticks(x + width * (len(FEATURES) - 1) / 2)
    ax.set_xticklabels(names, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f"{metric} (higher better)", fontsize=10)
axes.ravel()[0].legend(fontsize=7)

fig.suptitle(f"Classifiers (binder = pKd > {BINDER_PKD_THRESHOLD})", fontsize=12)
fig.tight_layout()
fig.savefig("models_structure_free/classifier_comparison.png", dpi=150)
plt.show()

## Save

In [ ]:
## Every model to its own pkl, plus the results table.
import joblib

for (fname, sname), (reg, clf) in trained.items():
    slug = f"{fname}__{sname}".replace("+", "_").replace("-", "_")
    joblib.dump(reg, f"models_structure_free/xgb_{slug}.pkl")
    joblib.dump(clf, f"models_structure_free/clf_{slug}.pkl")

results_df.to_csv("models_structure_free/results.csv", index=False)
print("Saved:", len(list(Path("models_structure_free").glob("*.pkl"))), "pkl files")
results_df